In [1]:
# !git clone https://github.com/valerizabby/coco-mulla.git
%cd coco-mulla

/content/coco-mulla


In [39]:
# !pip install -r /content/requirements.txt

# !pip install pydub
# !apt-get install -y ffmpeg

# !pip uninstall -y torch torchvision torchaudio torchao accelerate transformers huggingface_hub xformers

# !pip install torch==2.1.1 torchaudio==2.1.1 torchvision==0.16.1 --index-url https://download.pytorch.org/whl/cu118
# !pip install transformers==4.35.2
# !pip install huggingface-hub==0.23.0
# !pip install xformers==0.0.23.post1


# !pip uninstall -y torch torchaudio torchvision

# !pip install torch==2.1.1+cu118 torchaudio==2.1.1+cu118 torchvision==0.16.1+cu118 \
#     -f https://download.pytorch.org/whl/torch_stable.html

In [2]:
import zipfile
import os

zip_path = "/content/testset.zip"
extract_dir = "/content/unzipped_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

macosx_path = os.path.join(extract_dir, '__MACOSX')
if os.path.exists(macosx_path):
    import shutil
    shutil.rmtree(macosx_path)

print("Archive extracted")

Archive extracted


In [8]:
import os

def convert_lab_to_txt(lab_path, txt_path):
    with open(lab_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for i in range(len(lines)):
        parts = lines[i].strip().split()
        if len(parts) != 2:
            continue
        start = float(parts[0])
        chord = parts[1]
        if i + 1 < len(lines):
            end = float(lines[i + 1].strip().split()[0])
        else:
            end = start + 1.0  # добавим 1 секунду к последнему
        new_lines.append(f"{start:.3f}\t{end:.3f}\t{chord}\n")

    with open(txt_path, 'w') as f:
        f.writelines(new_lines)


def batch_convert_all_chords(testset_dir):
    for folder in sorted(os.listdir(testset_dir)):
        path = os.path.join(testset_dir, folder)
        if not os.path.isdir(path):
            continue
        lab_path = os.path.join(path, "chords.lab")
        txt_path = os.path.join(path, "chords.txt")
        if os.path.exists(lab_path):
            convert_lab_to_txt(lab_path, txt_path)
            print(f"✅ Converted: {folder}")
        else:
            print(f"⚠️ Skipped (no chords.lab): {folder}")


# Пример вызова
batch_convert_all_chords("/content/unzipped_data/testset/")

✅ Converted: track_001
✅ Converted: track_002
✅ Converted: track_003
✅ Converted: track_004
✅ Converted: track_005
✅ Converted: track_006
✅ Converted: track_007
✅ Converted: track_008
✅ Converted: track_009
✅ Converted: track_010


In [20]:
import re

def simplify_chord_label(label):
    if label == "N":
        return "N"
    m = re.match(r"([A-G][#b]?)(.*)", label)
    if not m:
        return None
    root, suffix = m.groups()
    suffix = suffix.lower()

    if "min" in suffix or "m" in suffix:
        return f"{root}:min"
    elif "7" in suffix:
        return f"{root}:7"
    else:
        return f"{root}:maj"

def fix_chord_labels(txt_path):
    fixed_lines = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) != 3:
                continue
            start, end, chord = parts
            fixed_chord = simplify_chord_label(chord)
            if fixed_chord is None:
                continue  # skip invalid
            fixed_lines.append(f"{start}\t{end}\t{fixed_chord}\n")

    with open(txt_path, "w") as f:
        f.writelines(fixed_lines)

def fix_all_chords(testset_dir):
    for folder in sorted(os.listdir(testset_dir)):
        path = os.path.join(testset_dir, folder)
        txt_path = os.path.join(path, "chords.txt")
        if os.path.exists(txt_path):
            fix_chord_labels(txt_path)
            print(f"✅ Fixed chords in {folder}")
        else:
            print(f"⚠️ Skipped {folder}, no chords.txt found.")

In [21]:
fix_all_chords("/content/unzipped_data/testset/")

✅ Fixed chords in track_001
✅ Fixed chords in track_002
✅ Fixed chords in track_003
✅ Fixed chords in track_004
✅ Fixed chords in track_005
✅ Fixed chords in track_006
✅ Fixed chords in track_007
✅ Fixed chords in track_008
✅ Fixed chords in track_009
✅ Fixed chords in track_010


In [31]:
from pydub import AudioSegment
import os

def convert_mp3_to_wav_in_folder(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".mp3"):
                mp3_path = os.path.join(root, file)
                wav_path = os.path.join(root, "audio.wav")
                if os.path.exists(wav_path):
                    continue  # уже сконвертировано
                print(f"🔄 Converting: {mp3_path} → {wav_path}")
                audio = AudioSegment.from_mp3(mp3_path)
                audio.export(wav_path, format="wav")

# Пример вызова
convert_mp3_to_wav_in_folder("/content/unzipped_data/testset")

🔄 Converting: /content/unzipped_data/testset/track_004/audio/TRDDDHM12903CC6E58.mp3 → /content/unzipped_data/testset/track_004/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_003/audio/TRCCCAQ128E079937C.mp3 → /content/unzipped_data/testset/track_003/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_007/audio/TRGGGDX128F4279B1F.mp3 → /content/unzipped_data/testset/track_007/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_010/audio/TRJJJAM128F425C444.mp3 → /content/unzipped_data/testset/track_010/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_006/audio/TRFFFFJ128F4259B34.mp3 → /content/unzipped_data/testset/track_006/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_002/audio/TRBBBWR12903CEE1D4.mp3 → /content/unzipped_data/testset/track_002/audio/audio.wav
🔄 Converting: /content/unzipped_data/testset/track_005/audio/TREEECE128F426A826.mp3 → /content/unzipped_data/testset/track_005/audio/audio.wav

In [36]:
!python inference.py \
  -o "/content/output" \
  -n 48 \
  -l 12 \
  -a "/content/unzipped_data/testset/track_001/audio/audio.wav" \
  -c "/content/unzipped_data/testset/track_001/chords.txt" \
  -m "/content/unzipped_data/testset/track_005/midi/c7a17fa29a93d2dd295de3053410d392.mid" \
  -e "/content/diff_9_end.pth" \
  -p "/content/unzipped_data/testset/track_001/prompt.txt" \
  -f 0

2025-05-25 16:59:42.510761: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-25 16:59:42.530427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748192382.552824   21154 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748192382.560049   21154 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-25 16:59:42.582276: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [38]:
import os
from tqdm import tqdm

# Импортируй свои функции здесь, если работаешь в отдельном файле
from coco_mulla.models import CoCoMulla
from coco_mulla.utilities import get_device, mkdir, read_lst, np2torch
from coco_mulla.utilities.encodec_utils import extract_rvq, save_rvq
from coco_mulla.utilities.symbolic_utils import process_midi, process_chord
from coco_mulla.utilities.sep_utils import separate
from config import TrainCfg

import torch
import torch.nn.functional as F
import numpy as np
import librosa

device = get_device()

def crop(x, mode, sample_sec, res, offset=0):
    xlen = x.shape[1] if mode in ["chord", "midi"] else x.shape[-1]
    sample_len = int(sample_sec * res) + 1
    if xlen < sample_len:
        if mode in ["chord", "midi"]:
            x = np.pad(x, ((0, 0), (0, sample_len - xlen), (0, 0)))
        else:
            x = F.pad(x, (0, sample_len - xlen), "constant", 0)
        return x
    st = offset * res
    ed = int((offset + sample_sec) * res) + 1
    return x[:, st: ed] if mode in ["chord", "midi"] else x[:, :, st: ed]

def load_data(audio_path, chord_path, midi_path, offset):
    sr, res, sample_sec = TrainCfg.sample_rate, TrainCfg.frame_res, TrainCfg.sample_sec
    wav, _ = librosa.load(audio_path, sr=sr, mono=True)
    wav = np2torch(wav).to(device)[None, None, ...]
    wavs = separate(wav, sr)
    drums_rvq = extract_rvq(wavs["drums"], sr=sr)
    chord, _ = process_chord(chord_path)
    midi, _ = process_midi(midi_path)

    chord = crop(chord[None, ...], "chord", sample_sec, res)
    pad_chord = chord.sum(-1, keepdims=True) == 0
    chord = np.concatenate([chord, pad_chord], -1)
    midi = crop(midi[None, ...], "midi", sample_sec, res, offset=offset)
    drums_rvq = crop(drums_rvq[None, ...], "drums_rvq", sample_sec, res, offset=offset)

    return drums_rvq.to(device).long(), torch.from_numpy(midi).to(device).float(), torch.from_numpy(chord).to(device).float()

def generate_mask(xlen):
    names = ["chord-only", "chord-drums", "chord-midi", "chord-drums-midi"]
    mask = torch.zeros([4, 2, xlen]).to(device)
    mask[0, 0] = 1                # Только аккорды
    mask[1, 1] = 1                # Только барабаны
    mask[2, 0] = 1                # Только миди (почему-то обозначено как chord)
    mask[3] = 1                   # Всё вместе
    return mask, names

def inference_on_all_tracks(base_path, model_path, num_layers=48, latent_dim=12, onset=0):
    model = CoCoMulla(TrainCfg.sample_sec, num_layers=num_layers, latent_dim=latent_dim).to(device)
    model.load_weights(model_path)
    model.eval()

    for folder in tqdm(sorted(os.listdir(base_path))):
        path = os.path.join(base_path, folder)
        if not os.path.isdir(path):
            continue

        print(f"🎧 Generating for {folder}...")
        try:
            audio_path = os.path.join(path, "audio", os.listdir(os.path.join(path, "audio"))[0])
            midi_path = os.path.join(path, "midi", os.listdir(os.path.join(path, "midi"))[0])
            chord_path = os.path.join(path, "chords.txt")
            prompt_path = os.path.join(path, "prompt.txt")

            drums_rvq, midi, chord = load_data(audio_path, chord_path, midi_path, offset=onset)
            cond_mask, names = generate_mask(drums_rvq.shape[-1])
            with open(prompt_path) as f:
                prompt = f.read().strip()

            batch = wrap_batch(drums_rvq, midi, chord, cond_mask, prompt)
            with torch.no_grad():
                pred = model(**batch)

            output_path = os.path.join(path, "cocomulla_output")
            mkdir(output_path)

            # Сохраняем каждый трек по его имени
            for i, name in enumerate(names):
                save_rvq([os.path.join(output_path, f"{name}.wav")], pred[i:i+1])

        except Exception as e:
            print(f"❌ Failed on {folder}: {e}")

# Пример вызова:
inference_on_all_tracks(
    base_path="/content/unzipped_data/testset",
    model_path="/content/diff_9_end.pth",
    num_layers=48,
    latent_dim=12,
    onset=0
)

/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


load....musicgen bk
lm_bk, here


  0%|          | 0/10 [00:00<?, ?it/s]

🎧 Generating for track_001...


CLIPPING /content/unzipped_data/testset/track_001/cocomulla_output/chord-only.wav happening with proba (a bit of clipping is okay): 0.0001296874979743734 maximum scale:  1.3670212030410767
CLIPPING /content/unzipped_data/testset/track_001/cocomulla_output/chord-drums.wav happening with proba (a bit of clipping is okay): 0.0007859374745748937 maximum scale:  1.591187596321106
CLIPPING /content/unzipped_data/testset/track_001/cocomulla_output/chord-midi.wav happening with proba (a bit of clipping is okay): 0.00026718751178123057 maximum scale:  1.3867542743682861
CLIPPING /content/unzipped_data/testset/track_001/cocomulla_output/chord-drums-midi.wav happening with proba (a bit of clipping is okay): 0.0024500000290572643 maximum scale:  1.555887222290039
 10%|█         | 1/10 [01:10<10:38, 70.96s/it]

...

CLIPPING /content/unzipped_data/testset/track_008/cocomulla_output/chord-only.wav happening with proba (a bit of clipping is okay): 0.014698437415063381 maximum scale:  2.3730881214141846
CLIPPING /content/unzipped_data/testset/track_008/cocomulla_output/chord-drums.wav happening with proba (a bit of clipping is okay): 0.004870312288403511 maximum scale:  1.9878840446472168
CLIPPING /content/unzipped_data/testset/track_008/cocomulla_output/chord-midi.wav happening with proba (a bit of clipping is okay): 0.010937499813735485 maximum scale:  1.8035662174224854
CLIPPING /content/unzipped_data/testset/track_008/cocomulla_output/chord-drums-midi.wav happening with proba (a bit of clipping is okay): 0.004554687533527613 maximum scale:  2.4204206466674805
 80%|████████  | 8/10 [09:28<02:22, 71.07s/it]

🎧 Generating for track_009...


CLIPPING /content/unzipped_data/testset/track_009/cocomulla_output/chord-only.wav happening with proba (a bit of clipping is okay): 0.0003937500005122274 maximum scale:  1.6585755348205566
CLIPPING /content/unzipped_data/testset/track_009/cocomulla_output/chord-drums.wav happening with proba (a bit of clipping is okay): 0.0012515624985098839 maximum scale:  1.3738605976104736
CLIPPING /content/unzipped_data/testset/track_009/cocomulla_output/chord-midi.wav happening with proba (a bit of clipping is okay): 0.0024984374176710844 maximum scale:  1.7320289611816406
CLIPPING /content/unzipped_data/testset/track_009/cocomulla_output/chord-drums-midi.wav happening with proba (a bit of clipping is okay): 0.0003953124978579581 maximum scale:  1.2177428007125854
 90%|█████████ | 9/10 [10:38<01:10, 70.95s/it]

🎧 Generating for track_010...


CLIPPING /content/unzipped_data/testset/track_010/cocomulla_output/chord-only.wav happening with proba (a bit of clipping is okay): 5.624999903375283e-05 maximum scale:  1.2275265455245972
CLIPPING /content/unzipped_data/testset/track_010/cocomulla_output/chord-drums.wav happening with proba (a bit of clipping is okay): 0.0003187500115018338 maximum scale:  1.3158479928970337
CLIPPING /content/unzipped_data/testset/track_010/cocomulla_output/chord-midi.wav happening with proba (a bit of clipping is okay): 0.010678124614059925 maximum scale:  2.5567197799682617
CLIPPING /content/unzipped_data/testset/track_010/cocomulla_output/chord-drums-midi.wav happening with proba (a bit of clipping is okay): 0.0041546872816979885 maximum scale:  2.2920727729797363
100%|██████████| 10/10 [11:50<00:00, 71.01s/it]


In [41]:
# !zip -r /content/testset.zip /content/unzipped_data/testset